<a href="https://colab.research.google.com/github/Mathildeholst/Speciale/blob/main/Fake_test_set.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Plapre-nano

##Download the Plapre-nano from Huggingface


In [ ]:
!pip install git+https://github.com/syv-ai/plapre.git
!pip install --upgrade "transformers>=4.51" "tokenizers>=0.21"

In [ ]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from huggingface_hub import snapshot_download
import json, os

model_dir = snapshot_download("syvai/plapre-nano")
config_path = os.path.join(model_dir, "tokenizer_config.json")
with open(config_path) as f:
    config = json.load(f)

config["tokenizer_class"] = "PreTrainedTokenizerFast"
config["extra_special_tokens"] = {}

with open(config_path, "w") as f:
    json.dump(config, f)

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

In [ ]:
import json, torch, numpy as np, soundfile as sf
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn as nn
from huggingface_hub import hf_hub_download
from kanade_tokenizer import KanadeModel, load_vocoder, vocode

tokenizer = AutoTokenizer.from_pretrained("syvai/plapre-nano")
model = AutoModelForCausalLM.from_pretrained(
    "syvai/plapre-nano", dtype=torch.bfloat16,
    device_map="auto", ignore_mismatched_sizes=True)

proj_path = hf_hub_download("syvai/plapre-nano", "speaker_proj.pt")
speaker_proj = nn.Linear(128, 960)
speaker_proj.load_state_dict(torch.load(proj_path, map_location="cpu"))
speaker_proj = speaker_proj.to(torch.bfloat16).eval()

import plapre
with open(Path(plapre.__file__).parent / "speakers.json") as f:
    speakers_raw = json.load(f)
speakers = {k: torch.tensor(v, dtype=torch.float32) for k, v in speakers_raw.items()}

device = torch.device("cuda")
kanade = KanadeModel.from_pretrained("frothywater/kanade-25hz-clean").eval().to(device)
vocoder_model = load_vocoder(kanade.config.vocoder_name).to(device)

def speak(text, speaker_name=None, temperature=0.8, top_p=0.95, top_k=50, max_new_tokens=500):
    spk_name = speaker_name or list(speakers.keys())[0]
    spk_emb = speakers[spk_name].to(device)
    with torch.no_grad():
        spk_hidden = speaker_proj(spk_emb.cpu().to(torch.bfloat16)).unsqueeze(0).unsqueeze(0)
    text_tag = tokenizer.convert_tokens_to_ids("<text>")
    audio_tag = tokenizer.convert_tokens_to_ids("<audio>")
    text_ids = tokenizer.encode(text, add_special_tokens=False)
    prompt_ids = [text_tag] + text_ids + [audio_tag]
    input_ids = torch.tensor([prompt_ids], device=device)
    with torch.no_grad():
        token_embeds = model.model.embed_tokens(input_ids)
        full_embeds = torch.cat([spk_hidden.to(device), token_embeds], dim=1)
    out = model.generate(
        inputs_embeds=full_embeds, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=temperature,
        top_p=top_p, top_k=top_k, eos_token_id=tokenizer.eos_token_id)
    audio_start = tokenizer.convert_tokens_to_ids("<audio_0>")
    audio_end = tokenizer.convert_tokens_to_ids("<audio_12799>")
    generated = out[0].tolist()
    kanade_indices = [t - audio_start for t in generated if audio_start <= t <= audio_end]
    if not kanade_indices:
        print("Ingen audio-tokens genereret!")
        return None
    tokens_tensor = torch.tensor(kanade_indices, dtype=torch.long, device=device)
    with torch.no_grad():
        mel = kanade.decode(content_token_indices=tokens_tensor, global_embedding=spk_emb.float().to(device))
        waveform = vocode(vocoder_model, mel.unsqueeze(0))
    audio = waveform.squeeze().cpu().numpy()
    sf.write("output.wav", audio, 24000)
    return audio

[2026-03-27 07:29:58,602] WARNING kanade_tokenizer: FlashAttention is not installed. Falling back to PyTorch SDPA implementation. There is no warranty that the model will work correctly.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

config.yaml: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/563M [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/torchaudio/models/wavlm_base_plus.pth" to /root/.cache/torch/hub/checkpoints/wavlm_base_plus.pth


100%|██████████| 360M/360M [00:06<00:00, 58.8MB/s]
[2026-03-27 07:30:17,292] INFO kanade_tokenizer: Loaded weights from safetensors file: /root/.cache/huggingface/hub/models--frothywater--kanade-25hz-clean/snapshots/cb2c8f10959ff0e5d3e97c9b82fcc3779c9532a5/model.safetensors
INFO:kanade_tokenizer:Loaded weights from safetensors file: /root/.cache/huggingface/hub/models--frothywater--kanade-25hz-clean/snapshots/cb2c8f10959ff0e5d3e97c9b82fcc3779c9532a5/model.safetensors


hift.pt:   0%|          | 0.00/83.4M [00:00<?, ?B/s]

Sound test

In [ ]:
audio = speak("Hej, hvordan har du det?", speaker_name="ida")

from IPython.display import Audio
Audio("output.wav")

Speaker distribution

In [ ]:
print(list(speakers.keys()))

['tor', 'ida', 'liv', 'ask', 'kaj']


##Voice cloning

In [ ]:
def speak(text, speaker_name=None, speaker_wav=None, temperature=0.8, top_p=0.95, top_k=50, max_new_tokens=500):

    if speaker_wav is not None:
        audio_data, sr = sf.read(speaker_wav)
        audio_tensor = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            result = kanade.encode(audio_tensor)
            spk_emb = result.global_embedding.squeeze()
    else:
        spk_name = speaker_name or list(speakers.keys())[0]
        spk_emb = speakers[spk_name].to(device)

    with torch.no_grad():
        spk_hidden = speaker_proj(spk_emb.cpu().to(torch.bfloat16)).unsqueeze(0).unsqueeze(0)

    text_tag = tokenizer.convert_tokens_to_ids("<text>")
    audio_tag = tokenizer.convert_tokens_to_ids("<audio>")
    text_ids = tokenizer.encode(text, add_special_tokens=False)
    prompt_ids = [text_tag] + text_ids + [audio_tag]
    input_ids = torch.tensor([prompt_ids], device=device)

    with torch.no_grad():
        token_embeds = model.model.embed_tokens(input_ids)
        full_embeds = torch.cat([spk_hidden.to(device), token_embeds], dim=1)

    out = model.generate(
        inputs_embeds=full_embeds, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=temperature,
        top_p=top_p, top_k=top_k, eos_token_id=tokenizer.eos_token_id)

    audio_start = tokenizer.convert_tokens_to_ids("<audio_0>")
    audio_end = tokenizer.convert_tokens_to_ids("<audio_12799>")
    generated = out[0].tolist()
    kanade_indices = [t - audio_start for t in generated if audio_start <= t <= audio_end]

    if not kanade_indices:
        print("Ingen audio-tokens genereret!")
        return None

    tokens_tensor = torch.tensor(kanade_indices, dtype=torch.long, device=device)
    with torch.no_grad():
        mel = kanade.decode(content_token_indices=tokens_tensor, global_embedding=spk_emb.float().to(device))
        waveform = vocode(vocoder_model, mel.unsqueeze(0))

    audio = waveform.squeeze().cpu().numpy()
    sf.write("output.wav", audio, 24000)
    return audio

Mathilde

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!ffmpeg -i "Mathilde.m4a" -ar 24000 -ac 1 mathilde.wav -y

audio = speak("mit navn er Mathilde, jeg er 25 år gammel. jeg bor til hverdag i århus", speaker_wav="mathilde.wav")

from IPython.display import Audio
Audio("output.wav")

In [ ]:
import numpy as np
audio_data, sr = sf.read("mathilde.wav")
audio_tensor = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0).to(device)
with torch.no_grad():
    result = kanade.encode(audio_tensor)
    spk_emb = result.global_embedding.squeeze()

speakers["mathilde"] = spk_emb.cpu()

print(list(speakers.keys()))

['tor', 'ida', 'liv', 'ask', 'kaj', 'mathilde']


Niels

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!ffmpeg -i "Niels.m4a" -ar 24000 -ac 1 niels.wav -y

audio = speak("mit navn er Niels, jeg er 25 år gammel, jeg bor til hverdag i århus", speaker_wav="niels.wav")

from IPython.display import Audio
Audio("output.wav")

In [ ]:
audio_data, sr = sf.read("niels.wav")
audio_tensor = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0).to(device)

with torch.no_grad():
    result = kanade.encode(audio_tensor)
    spk_emb = result.global_embedding.squeeze()

speakers["niels"] = spk_emb.cpu()

In [ ]:
print(list(speakers.keys()))

['tor', 'ida', 'liv', 'ask', 'kaj', 'mathilde', 'niels']


Clara

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!ffmpeg -i "Clara.m4a" -ar 24000 -ac 1 clara.wav -y

audio = speak("mit navn er clara, jeg er 25 år gammel, jeg bor til hverdag i århus", speaker_wav="clara.wav")

from IPython.display import Audio
Audio("output.wav")

In [ ]:
audio_data, sr = sf.read("clara.wav")
audio_tensor = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0).to(device)

with torch.no_grad():
    result = kanade.encode(audio_tensor)
    spk_emb = result.global_embedding.squeeze()

speakers["clara"] = spk_emb.cpu()

In [ ]:
print(list(speakers.keys()))

['tor', 'ida', 'liv', 'ask', 'kaj', 'mathilde', 'niels', 'clara']


Freja

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!ffmpeg -i "Freja.m4a" -ar 24000 -ac 1 freja.wav -y

audio = speak("mit navn er Freja, jeg er 25 år gammel, jeg bor til hverdag i århus", speaker_wav="freja.wav")

from IPython.display import Audio
Audio("output.wav")

In [ ]:
audio_data, sr = sf.read("freja.wav")
audio_tensor = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0).to(device)

with torch.no_grad():
    result = kanade.encode(audio_tensor)
    spk_emb = result.global_embedding.squeeze()

speakers["freja"] = spk_emb.cpu()

In [ ]:
print(list(speakers.keys()))

['tor', 'ida', 'liv', 'ask', 'kaj', 'mathilde', 'niels', 'clara', 'freja']


Emil

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!ffmpeg -i "Emil.mp3" -ar 24000 -ac 1 emil.wav -y

audio = speak("mit navn er Emil, jeg er 25 år gammel, jeg bor til hverdag i århus", speaker_wav="emil.wav")

from IPython.display import Audio
Audio("output.wav")

In [ ]:
audio_data, sr = sf.read("emil.wav")
audio_tensor = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0).to(device)

with torch.no_grad():
    result = kanade.encode(audio_tensor)
    spk_emb = result.global_embedding.squeeze()

speakers["emil"] = spk_emb.cpu()

In [ ]:
print(list(speakers.keys()))

['tor', 'ida', 'liv', 'ask', 'kaj', 'mathilde', 'niels', 'clara', 'freja', 'emil']


Martin

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!ffmpeg -i "Martin.m4a" -ar 24000 -ac 1 martin.wav -y

audio = speak("mit navn er Martin, jeg er 25 år gammel, jeg bor til hverdag i århus", speaker_wav="martin.wav")

from IPython.display import Audio
Audio("output.wav")

In [ ]:
audio_data, sr = sf.read("martin.wav")
audio_tensor = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0).to(device)

with torch.no_grad():
    result = kanade.encode(audio_tensor)
    spk_emb = result.global_embedding.squeeze()

speakers["martin"] = spk_emb.cpu()

In [ ]:
print(list(speakers.keys()))

['tor', 'ida', 'liv', 'ask', 'kaj', 'mathilde', 'niels', 'clara', 'freja', 'emil', 'martin']


Franciska

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!ffmpeg -i "Franciska.m4a" -ar 24000 -ac 1 franciska.wav -y

audio = speak("mit navn er Franciska, jeg er 25 år gammel, jeg bor til hverdag i århus", speaker_wav="franciska.wav")

from IPython.display import Audio
Audio("output.wav")

In [ ]:
audio_data, sr = sf.read("franciska.wav")
audio_tensor = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0).to(device)

with torch.no_grad():
    result = kanade.encode(audio_tensor)
    spk_emb = result.global_embedding.squeeze()

speakers["franciska"] = spk_emb.cpu()

In [ ]:
print(list(speakers.keys()))

['tor', 'ida', 'liv', 'ask', 'kaj', 'mathilde', 'niels', 'clara', 'freja', 'emil', 'martin', 'franciska']


Test voice cloning

In [ ]:
audio = speak("Hej, hvordan har du det?", speaker_name="ida")

from IPython.display import Audio
Audio("output.wav")

##Generation of fake test set

Upload test csv

In [ ]:
from google.colab import files
files.upload()

In [ ]:
print(type(speakers))
print(speakers)

<class 'list'>
['tor', 'ida', 'liv', 'ask', 'kaj', 'mathilde', 'niels', 'clara', 'freja', 'emil', 'martin', 'franciska']


In [ ]:
import pandas as pd

test_df = pd.read_csv("test.csv", sep=';')
test_sentences = test_df[['sætning_id', 'tekst']].drop_duplicates()

print(f"Antal unikke sætninger i test: {len(test_sentences)}")
print(f"Antal stemmer: {len(speakers)}")
print(len(test_df['tekst']))

Antal unikke sætninger i test: 343
Antal stemmer: 12
1100


Add cloned voices to Plapre-nano

In [ ]:
import plapre
from pathlib import Path

with open(Path(plapre.__file__).parent / "speakers.json") as f:
    speakers_raw = json.load(f)
speakers = {k: torch.tensor(v, dtype=torch.float32) for k, v in speakers_raw.items()}

print(type(speakers))
print(list(speakers.keys()))

<class 'dict'>
['tor', 'ida', 'liv', 'ask', 'kaj']


In [ ]:
klonede_stemmer = ["mathilde", "niels", "clara", "freja", "emil", "martin", "franciska"]

for navn in klonede_stemmer:
    wav_fil = f"{navn}.wav"
    if os.path.exists(wav_fil):
        audio_data, sr = sf.read(wav_fil)
        audio_tensor = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            result = kanade.encode(audio_tensor)
            spk_emb = result.global_embedding.squeeze()
        speakers[navn] = spk_emb.cpu()
        print(f"Tilfojet: {navn}")
    else:
        print(f"Fil ikke fundet: {wav_fil}")

print(f"Stemmer klar: {list(speakers.keys())}")

Tilfojet: mathilde
Tilfojet: niels
Tilfojet: clara
Tilfojet: freja
Tilfojet: emil
Tilfojet: martin
Tilfojet: franciska
Stemmer klar: ['tor', 'ida', 'liv', 'ask', 'kaj', 'mathilde', 'niels', 'clara', 'freja', 'emil', 'martin', 'franciska']


In [ ]:
import os
import soundfile as sf
import pandas as pd

os.makedirs("audio_data/test/fake", exist_ok=True)

fake_data = []
total = len(test_df)
count = 0

for _, row in test_df.iterrows():
    sentence_id = row['sætning_id']
    text = row['tekst']

    speaker = list(speakers.keys())[count % len(speakers)]

    filename = f"{speaker}_{sentence_id}_{count}.wav"
    path = f"audio_data/test/fake/{filename}"

    generated_audio = speak(text, speaker_name=speaker)

    if generated_audio is not None:
        sf.write(path, generated_audio, 24000)

    fake_data.append({
        "id": filename,
        "tekst": text,
        "sætning_id": sentence_id,
        "speaker": speaker,
        "label": "fake"
    })

    count += 1
    if count % 50 == 0:
        print(f"{count}/{total} filer genereret...")

print(f"Faerdigt! {count} filer genereret")

50/1100 filer genereret...
100/1100 filer genereret...
150/1100 filer genereret...
200/1100 filer genereret...
250/1100 filer genereret...
300/1100 filer genereret...
350/1100 filer genereret...
400/1100 filer genereret...
450/1100 filer genereret...
500/1100 filer genereret...
550/1100 filer genereret...
600/1100 filer genereret...
650/1100 filer genereret...
700/1100 filer genereret...
750/1100 filer genereret...
800/1100 filer genereret...
850/1100 filer genereret...
900/1100 filer genereret...
950/1100 filer genereret...
1000/1100 filer genereret...
1050/1100 filer genereret...
1100/1100 filer genereret...
Faerdigt! 1100 filer genereret


In [ ]:
import shutil
from google.colab import files

fake_df = pd.DataFrame(fake_data)
fake_df.to_csv("test_fake.csv", index=False)

shutil.make_archive("fake_test_audio", 'zip', "audio_data/test/fake")
files.download("fake_test_audio.zip")
files.download("test_fake.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.upload()

import pandas as pd
df = pd.read_csv("test_fake.csv")

print(f"Antal filer i alt: {len(df)}")
print(f"\nFordeling per stemme:")
print(df['speaker'].value_counts().to_string())

Saving test_fake.csv to test_fake.csv
Antal filer i alt: 1100

Fordeling per stemme:
speaker
tor          92
ida          92
liv          92
ask          92
kaj          92
mathilde     92
niels        92
clara        92
freja        91
emil         91
martin       91
franciska    91
